In [ ]:
from types import SimpleNamespace
import warnings
from tensorflow.keras.callbacks import ModelCheckpoint
import tensorflow as tf

from utils import create_new_experiment
from callbacks import WarmupSchedule, SamplesSeen
from constants import DATA_ROOT, RESULTS_ROOT
from dataset import Dataset
from model import RNABagModel

In [ ]:
# tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
warnings.filterwarnings(
    "ignore",
    message="Observation names are not unique"
)

In [ ]:
# CONFIGURE RUN

CONFIG = {
    'use_wandb': True,  # Set True after running wandb login in the terminal.
    'project': 'scUNVEIL Training',
    'run_name': '',

    'emb_dim': 2048,
    'ff_dim': 2048,
    'n_layers': 16,

    'n_genes': 60_000,
    'backed_r': True,

    'lr': 1e-4,

    'warmup_steps': 10_000,
    'total_steps': 1_000_000,

    'batch_size': 1000,

    'steps_per_epoch': 10_000,
    'limit_loaded_shards': None,
}
CF = SimpleNamespace(**CONFIG)

In [ ]:
# TRACKING & CHECKPOINT DIR INIT

# Create Result Directory
experiment_id = create_new_experiment(CONFIG)

# Use credentials established by wandb login in the terminal.
# WANDB_ENTITY can optionally select the destination team/account.
wandb_run = None
use_wandb = CF.use_wandb

if use_wandb:
    import wandb

    wandb_run = wandb.init(
        project=CF.project,
        name=f"emb={CF.emb_dim} l={CF.n_layers} g={int(round(CF.n_genes / 1000))}k lr={CF.lr:.0e} bs={CF.batch_size}",
        config=CONFIG,
        id=experiment_id,
    )


In [ ]:
# DATASET INIT

data = Dataset(
    source_dir=DATA_ROOT / 'shuffled_shards', 
    backed_r=CF.backed_r, 
    limit_loaded_shards=CF.limit_loaded_shards,
)

dg_train = data.pretrain_dg(CF.batch_size, ds='train', max_genes=CF.n_genes)

In [ ]:
# MODEL INIT

m = RNABagModel(n_vars=CF.n_genes, n_layers=CF.n_layers, emb_dim=CF.emb_dim, ff_dim=CF.ff_dim)

lr_schedule = WarmupSchedule(
    base_lr=CF.lr, 
    warmup_steps=CF.warmup_steps, 
    finish_lr=CF.lr / 100.0, 
    decay_steps=CF.total_steps - CF.warmup_steps
)

m.compile_pretrain(lr_schedule)

In [ ]:
# TRAINING LOOP

train_callbacks = [
    SamplesSeen(batch_size=CF.batch_size),
    ModelCheckpoint(
        str(RESULTS_ROOT / 'training' / experiment_id / 'weights' / '{epoch:04d}.weights.h5'),
        save_best_only=False,
        save_weights_only=True,
    )
]

if use_wandb:
    from wandb.integration.keras import WandbMetricsLogger
    train_callbacks.append(WandbMetricsLogger())

m.model.fit(
    dg_train,
    epochs=CF.total_steps // CF.steps_per_epoch,
    callbacks=train_callbacks,
    steps_per_epoch=CF.steps_per_epoch
)